In [100]:
import av
import os

In [126]:
input_path = "./ffmpeg_practice/input/naruto_clip.mp4"
output_path = "./ffmpeg_practice/output/cut_clip.mp4"

if os.path.exists(input_path):
    print(f"File found: {input_path}")
else:
    print(f"File not found.")
av.logging.set_level(av.logging.VERBOSE)

container = av.open(input_path)
container

File found: ./ffmpeg_practice/input/naruto_clip.mp4


<av.InputContainer './ffmpeg_practice/input/naruto_clip.mp4'>

In [127]:
stream = container.streams.video[0]
stream

<av.VideoStream #0 h264, yuv420p 854x480 at 0x1f1bb7583b0>

In [128]:
fps = float(stream.average_rate)
fps

23.976023976023978

In [129]:
import math
for index, frame in enumerate(container.decode(stream)):
    print(f"Index: {index}")
    print(f"Index / fps: {index / fps}")
    if index % math.ceil(fps) == 0:
        frame.to_image().save(f"{output_path}_{index}.png")

Index: 0
Index / fps: 0.0
Index: 1
Index / fps: 0.04170833333333333
Index: 2
Index / fps: 0.08341666666666667
Index: 3
Index / fps: 0.125125
Index: 4
Index / fps: 0.16683333333333333
Index: 5
Index / fps: 0.20854166666666665
Index: 6
Index / fps: 0.25025
Index: 7
Index / fps: 0.2919583333333333
Index: 8
Index / fps: 0.33366666666666667
Index: 9
Index / fps: 0.37537499999999996
Index: 10
Index / fps: 0.4170833333333333
Index: 11
Index / fps: 0.45879166666666665
Index: 12
Index / fps: 0.5005
Index: 13
Index / fps: 0.5422083333333333
Index: 14
Index / fps: 0.5839166666666666
Index: 15
Index / fps: 0.625625
Index: 16
Index / fps: 0.6673333333333333
Index: 17
Index / fps: 0.7090416666666666
Index: 18
Index / fps: 0.7507499999999999
Index: 19
Index / fps: 0.7924583333333333
Index: 20
Index / fps: 0.8341666666666666
Index: 21
Index / fps: 0.875875
Index: 22
Index / fps: 0.9175833333333333
Index: 23
Index / fps: 0.9592916666666667
Index: 24
Index / fps: 1.001
Index: 25
Index / fps: 1.042708333

In [ ]:
input = av.open(input_path, mode="r")
output = av.open(f'{output_path}-remuxed.mp4', mode="w")

input_stream = input.streams.video[0]
output_stream = output.add_stream(input_stream.codec_context.name)

: 

In [ ]:
import av
from pathlib import Path

input_container = av.open(input_path, mode="r")

p = Path(output_path)
remuxed_path = p.with_name(p.stem + "-remuxed" + p.suffix)

output_container = av.open(str(remuxed_path), mode="w")

input_stream = input_container.streams.video[0]

output_stream = output_container.add_stream(
    input_stream.codec_context.name,
    rate=input_stream.average_rate,
)

output_stream.time_base = input_stream.time_base
output_stream.codec_context.extradata = input_stream.codec_context.extradata

for packet in input_container.demux(input_stream):
    if packet.dts is None:
        continue

    packet.stream = output_stream
    packet.pts = packet.pts
    packet.dts = packet.dts
    packet.time_base = input_stream.time_base

    output_container.mux(packet)

output_container.close()
input_container.close()

./ffmpeg_practice/output/cut_clip.mp4
ffmpeg_practice\output\cut_clip-remuxed.mp4
